# Qwen3-0.6B-Base + LoRA on GSM8K

Simple Kaggle notebook for:

1. Loading `Qwen/Qwen3-0.6B-Base`
2. LoRA fine-tuning on the GSM8K training split
3. Training loss only on the GSM8K solution/answer tokens
4. Greedy evaluation on the untouched GSM8K test split
5. Using the same `Final Answer: <number>` format as the zero-shot notebook

**LoRA configuration:** rank 16, alpha 32, dropout 0.05, all main Qwen linear projections, 3 epochs, learning rate `2e-4`.

In [1]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 72.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.1 MB/s eta 0:00:00:00:01


In [2]:
import os
import re
import time
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
OUTPUT_DIR = "/kaggle/working/qwen3_0.6b_gsm8k_lora"
ADAPTER_DIR = "/kaggle/working/qwen3_0.6b_gsm8k_lora_adapter"

MAX_LENGTH = 512
MAX_NEW_TOKENS = 512

# Set to an integer such as 1000 for a quick test run.
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
BF16 supported: True


## 1. Load tokenizer and base model

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Right padding for training.
tokenizer.padding_side = "right"

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
else:
    MODEL_DTYPE = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())

print("Model:", MODEL_NAME)
print(f"Total parameters: {total_params:,}")
print(f"Total parameters: {total_params / 1e6:.2f}M")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model: Qwen/Qwen3-0.6B-Base
Total parameters: 596,049,920
Total parameters: 596.05M


## 2. Load GSM8K

The official `train` split is used only for training and the official `test` split is kept untouched for final evaluation.

In [5]:
dataset = load_dataset("openai/gsm8k", "main")

train_dataset = dataset["train"]
test_dataset = dataset["test"]

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(
        range(min(MAX_TRAIN_SAMPLES, len(train_dataset)))
    )

if MAX_TEST_SAMPLES is not None:
    test_dataset = test_dataset.select(
        range(min(MAX_TEST_SAMPLES, len(test_dataset)))
    )

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

print("\nExample question:")
print(train_dataset[0]["question"])

print("\nOriginal GSM8K solution:")
print(train_dataset[0]["answer"])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train examples: 7473
Test examples: 1319

Example question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Original GSM8K solution:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## 3. Use the same prompt and answer format as the zero-shot notebook

In [6]:
def create_prompt(question):
    return f'''Solve the following math problem step by step.

Question:
{question}

At the end, write the final numerical answer exactly in this format:

Final Answer: <number>

Answer:
'''


def extract_gold_answer(answer_text):
    match = re.search(
        r"####\s*([-+]?[0-9,]*\.?[0-9]+)",
        answer_text
    )
    if match:
        return match.group(1).replace(",", "")
    return None


def create_training_answer(answer_text):
    # Preserve GSM8K reasoning but use the same final-answer format
    # that the zero-shot evaluator expects.
    gold = extract_gold_answer(answer_text)

    if gold is None:
        return answer_text

    reasoning = answer_text.rsplit("####", 1)[0].strip()

    return f"{reasoning}\n\nFinal Answer: {gold}"

In [7]:
question = train_dataset[0]["question"]
target = create_training_answer(train_dataset[0]["answer"])

print("PROMPT")
print(create_prompt(question))

print("\nTRAINING TARGET")
print(target)

PROMPT
Solve the following math problem step by step.

Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

At the end, write the final numerical answer exactly in this format:

Final Answer: <number>

Answer:


TRAINING TARGET
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.

Final Answer: 72


## 4. Tokenize for answer-only loss

Prompt tokens receive label `-100`, so cross-entropy loss is computed only on the worked solution and final answer.

In [8]:
def tokenize_example(example):
    prompt = create_prompt(example["question"])
    target = create_training_answer(example["answer"])

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False
    )["input_ids"]

    target_ids = target_ids + [tokenizer.eos_token_id]

    # Always leave at least one token for the target.
    if len(prompt_ids) >= MAX_LENGTH:
        prompt_ids = prompt_ids[:MAX_LENGTH - 1]

    room_for_target = MAX_LENGTH - len(prompt_ids)

    # If needed, keep the end of the solution so Final Answer remains.
    if len(target_ids) > room_for_target:
        target_ids = target_ids[-room_for_target:]

    input_ids = prompt_ids + target_ids
    labels = ([-100] * len(prompt_ids)) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing GSM8K train split",
)

print(tokenized_train)
print("First tokenized length:", len(tokenized_train[0]["input_ids"]))
print(
    "First example supervised tokens:",
    sum(x != -100 for x in tokenized_train[0]["labels"])
)

Tokenizing GSM8K train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7473
})
First tokenized length: 135
First example supervised tokens: 62


## 5. Add LoRA

This is normal FP16/BF16 LoRA. The base model is **not** 4-bit quantized.

In [9]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
Trainable parameters: 10,092,544
Trainable percentage: 1.6932%


## 6. Train LoRA

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

use_bf16 = bool(
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
)
use_fp16 = bool(
    torch.cuda.is_available() and not use_bf16
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=20,
    lr_scheduler_type="cosine",

    logging_steps=20,

    save_strategy="epoch",
    save_total_limit=1,

    gradient_checkpointing=True,

    bf16=use_bf16,
    fp16=use_fp16,

    optim="adamw_torch",
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

train_result = trainer.train()

print("\nTraining finished")
print("Training loss:", train_result.training_loss)

Step,Training Loss
20,0.516261
40,0.440754
60,0.422509
80,0.413086
100,0.407132
120,0.398437
140,0.375281
160,0.374918
180,0.364325
200,0.369235



Training finished
Training loss: 0.37787485122680664


## 7. Save the LoRA adapter

In [11]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("LoRA adapter saved to:")
print(ADAPTER_DIR)

LoRA adapter saved to:
/kaggle/working/qwen3_0.6b_gsm8k_lora_adapter


## 8. Test one GSM8K example

In [12]:
model = trainer.model
model.eval()
model.config.use_cache = True

# Left padding is important for batched decoder-only generation.
tokenizer.padding_side = "left"

def extract_model_answer(text):
    match = re.search(
        r"Final Answer:\s*([-+]?[0-9,]*\.?[0-9]+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).replace(",", "")

    return None


question = test_dataset[0]["question"]
prompt = create_prompt(question)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

gold_answer = extract_gold_answer(test_dataset[0]["answer"])
predicted_answer = extract_model_answer(response)

print("QUESTION")
print(question)

print("\nMODEL RESPONSE")
print(response)

print("\nPREDICTED ANSWER:", predicted_answer)
print("GOLD ANSWER:", gold_answer)
print("CORRECT:", predicted_answer == gold_answer)

QUESTION
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL RESPONSE
Janet's ducks lay 16 eggs per day, so she has 16 * 3 = <<16*3=48>>48 fresh duck eggs.
She eats 3 eggs for breakfast every morning, so she has 48 - 3 = <<48-3=45>>45 fresh duck eggs left.
She bakes 4 muffins for her friends every day, so she has 45 - 4 = <<45-4=41>>41 fresh duck eggs left.
She sells the remaining 41 fresh duck eggs at $2 per egg, so she makes 41 * 2 = $<<41*2=82>>82.

Final Answer: 82

PREDICTED ANSWER: 82
GOLD ANSWER: 18
CORRECT: False


## 9. Evaluate on the full GSM8K test set

This matches the zero-shot notebook's exact-answer evaluation.

In [13]:
BATCH_SIZE = 32

results = []
correct = 0
valid_count = 0

start_time = time.time()

for start_idx in tqdm(
    range(0, len(test_dataset), BATCH_SIZE),
    desc="Evaluating LoRA model"
):
    end_idx = min(start_idx + BATCH_SIZE, len(test_dataset))

    batch = test_dataset.select(range(start_idx, end_idx))

    questions = batch["question"]
    gold_solutions = batch["answer"]

    prompts = [
        create_prompt(question)
        for question in questions
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[
        :,
        inputs["input_ids"].shape[1]:
    ]

    responses = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    for question, gold_solution, response in zip(
        questions,
        gold_solutions,
        responses
    ):
        gold_answer = extract_gold_answer(gold_solution)
        predicted_answer = extract_model_answer(response)

        valid_format = predicted_answer is not None
        is_correct = predicted_answer == gold_answer

        if valid_format:
            valid_count += 1

        if is_correct:
            correct += 1

        results.append({
            "question": question,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "valid_format": valid_format,
            "response": response,
        })

elapsed_time = time.time() - start_time

accuracy = correct / len(test_dataset)
valid_format_rate = valid_count / len(test_dataset)

print("\n================================")
print("LoRA-SFT GSM8K RESULT")
print("================================")
print("Base model:", MODEL_NAME)
print("Samples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

print(f"Correct: {correct}/{len(test_dataset)}")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Valid format rate: {valid_format_rate * 100:.2f}%")

print(f"Evaluation time: {elapsed_time / 60:.2f} minutes")
print(
    f"Average time/question: "
    f"{elapsed_time / len(test_dataset):.3f} seconds"
)

Evaluating LoRA model:   0%|          | 0/42 [00:00<?, ?it/s]


LoRA-SFT GSM8K RESULT
Base model: Qwen/Qwen3-0.6B-Base
Samples: 1319
Batch size: 32
Correct: 669/1319
Accuracy: 50.72%
Valid format rate: 98.56%
Evaluation time: 50.63 minutes
Average time/question: 2.303 seconds


## 10. Save predictions and experiment summary

In [14]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "/kaggle/working/qwen3_0.6b_gsm8k_lora_predictions.csv",
    index=False
)

summary = {
    "experiment": "lora_sft",
    "model": MODEL_NAME,
    "dataset": "GSM8K",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "epochs": 3,
    "learning_rate": 2e-4,
    "max_length": MAX_LENGTH,
    "num_train_samples": len(train_dataset),
    "num_test_samples": len(test_dataset),
    "correct": correct,
    "accuracy": accuracy,
    "valid_format_rate": valid_format_rate,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "training_loss": train_result.training_loss,
    "evaluation_seconds": elapsed_time,
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    "/kaggle/working/qwen3_0.6b_gsm8k_lora_summary.csv",
    index=False
)

display(summary_df)

,experiment,model,dataset,lora_r,lora_alpha,lora_dropout,epochs,learning_rate,max_length,num_train_samples,num_test_samples,correct,accuracy,valid_format_rate,total_parameters,trainable_parameters,training_loss,evaluation_seconds
0,lora_sft,Qwen/Qwen3-0.6B-Base,GSM8K,16,32,0.05,3,0.0002,512,7473,1319,669,0.507202,0.985595,596049920,10092544,0.377875,3037.649681


In [15]:
incorrect_df = results_df[results_df["correct"] == False]

print("Incorrect examples:", len(incorrect_df))

display(
    incorrect_df[
        [
            "question",
            "gold_answer",
            "predicted_answer",
            "response",
        ]
    ].head(10)
)

Incorrect examples: 650


,question,gold_answer,predicted_answer,response
0,Janet’s ducks lay 16 eggs per day. She eats th...,18,82,"Janet's ducks lay 16 eggs per day, so she has ..."
2,Josh decides to try flipping a house. He buys...,70000,120000,"The house increased in value by 150/100*$80,00..."
3,James decides to run 3 sprints 3 times a week....,540,180,He runs 3*60=<<3*60=180>>180 meters a week\n\n...
4,"Every day, Wendi feeds each of her chickens th...",20,60,Wendi feeds her chickens 3 cups of feed per me...
7,Carla is downloading a 200 GB file. Normally s...,160,60,First find the download speed after the restar...
8,John drives for 3 hours at a speed of 60 mph a...,45,80,He drives for 3 hours at 60 mph for a distance...
10,A new program had 60 downloads in the first mo...,366,348,The number of downloads in the second month wa...
11,Toula went to the bakery and bought various ty...,694,794,The cost of the donuts is 3 x $68 = $<<3*68=20...
12,Carlos is planting a lemon tree. The tree will...,13,1,"The lemon tree will grow 7 lemons per year, an..."
13,Melanie is a door-to-door saleswoman. She sold...,18,15,Melanie sold 5*2=<<5*2=10>>10 vacuum cleaners ...
